# ArmyOfSafeguards — Colab GPU SFT (experts)

This notebook fine-tunes one expert model on **native curriculum JSONL** and logs metrics to `training/experts/sft_metrics.jsonl`.

**Prereqs**
- Colab runtime: **GPU**
- Optional: Hugging Face token for gated datasets (`HF_TOKEN`)

In [ ]:
# (Colab) Check GPU
!nvidia-smi

import sys, torch
print('python', sys.version)
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())

In [ ]:
# (Colab) Clone repo + install deps
# If you're running from your own fork, change the URL.

REPO_URL = "https://github.com/SohamNagi/ArmyOfSafeguards.git"
REPO_DIR = "ArmyOfSafeguards"

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR

!pip -q install -r requirements.txt
# Optional (only needed if you pass --lora):
!pip -q install peft

In [ ]:
# (Optional) Hugging Face login for gated datasets
# Option A: set HF_TOKEN in Colab "Secrets" then run this cell.
import os

hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
else:
    print("HF_TOKEN not set. If you hit gated dataset errors, set HF_TOKEN and rerun.")

In [ ]:
# Build native curriculum SFT JSONL (choose one expert)
# Options for EXPERT: jailbreak | toxicity | sexual | factuality

EXPERT = "sexual"
OUT_JSONL = f"/content/{EXPERT}_native.jsonl"  # put under /content for easy download

!python training/experts/build_expert_sft_jsonl.py --expert "$EXPERT" --out "$OUT_JSONL"
!python -c "import json; p='$OUT_JSONL'; print('jsonl', p); print('rows', sum(1 for _ in open(p, 'r', encoding='utf-8')) )"

In [ ]:
# Run SFT (writes metrics to training/experts/sft_metrics.jsonl by default)
# Tip: add --fp16 for A100/T4, or --bf16 for A100.

OUTPUT_DIR = f"experts/artifacts/{EXPERT}_ft"

# Domain-specific wrappers exist, but we call the shared trainer directly so we can swap args easily.
!python training/common/sequence_classifier_train.py \
  --data "$OUT_JSONL" \
  --domain "$EXPERT" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 2 \
  --batch 16 \
  --grad-accum 1 \
  --lr 2e-5 \
  --max-length 256 \
  --fp16

print("Saved model to", OUTPUT_DIR)
print("Runtime env var:")
print({
    "toxicity": "AOS_TOXICITY_MODEL",
    "sexual": "AOS_SEXUAL_MODEL",
    "factuality": "AOS_FACTUALITY_MODEL",
    "jailbreak": "AOS_JAILBREAK_MODEL",
}.get(EXPERT, "AOS_<EXPERT>_MODEL"), "=", OUTPUT_DIR)

In [ ]:
# View / compare runs
!ls -lh training/experts/sft_metrics.jsonl || true
!python training/experts/summarize_sft_metrics.py training/experts/sft_metrics.jsonl